In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 1. Open the correlation GeoTIFF (Update this to your actual file path)
corr_file = "/content/Temperature Vs Ozon.tif"

with rasterio.open(corr_file) as src:
    corr_data = src.read(1).astype(float)
    if src.nodata is not None:
        corr_data[corr_data == src.nodata] = np.nan

    bounds = src.bounds
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    spatial_mean_corr = np.nanmean(corr_data)

# 2. Setup a Fixed, Diverging Discrete Palette (-1 to +1)
# 8 or 10 bins work perfectly for showing negative vs positive spatial splits
num_classes = 10
bounds_ticks = np.linspace(-1.0, 1.0, num_classes + 1)

# 'RdBu_r' is the cartographic standard for correlation (Red=Positive, Blue=Negative)
base_cmap = plt.colormaps['RdBu_r']
cmap = mcolors.ListedColormap(base_cmap(np.linspace(0, 1, num_classes)))
norm = mcolors.BoundaryNorm(bounds_ticks, cmap.N)

# 3. Plotting the Single Map
fig, ax = plt.subplots(figsize=(12, 5))

im = ax.imshow(corr_data, cmap=cmap, norm=norm, extent=extent, aspect='auto')
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

# Titles & Meta Formatting
ax.set_title(f"Spatial Correlation: Temperature vs. Ozone", fontsize=12, pad=12)
ax.tick_params(labelsize=9)

# Add your Dotted Grid Lines on Top
ax.grid(True, which='major', linestyle=':', color='black', alpha=0.5, linewidth=1)
ax.set_axisbelow(False)

# 4. Add the Colorbar explicitly mapped to Correlation Space
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', shrink=0.5, aspect=35, pad=0.15, ticks=bounds_ticks)
cbar.set_label('Correlation Coefficient ($r$)', fontsize=11, labelpad=8)
cbar.ax.tick_params(labelsize=9)
cbar.ax.set_xticklabels([f"{x:.1f}" for x in bounds_ticks])

plt.tight_layout()
plt.show()
